In [10]:
from pathlib import Path
import cv2
import numpy as np 
from ultralytics import YOLO
model = YOLO("yolo26n-seg.pt")
results = model.predict(source="max.jpg")
for r in results:
    img = np.copy(r.orig_img)
    img_name = Path(r.path).stem # source image base-name
    for ci, c in enumerate(r):
        label = c.names[c.boxes.cls.tolist().pop()] #class name
        b_mask = np.zeros(img.shape[:2], np.uint8)
        contur = c.masks.xy[0].astype(np.int32).reshape(-1, 1, 2)
        cv2.drawContours(b_mask, [contur], -1, (255, 255, 255), cv2.FILLED)
        mask3ch = cv2.cvtColor(b_mask, cv2.COLOR_GRAY2BGR)
        isolated = cv2.bitwise_and(mask3ch, img)
        isolated = np.dstack([img,b_mask])
        x1, y1, x2, y2 = c.boxes.xyxy.cpu().numpy().squeeze().astype(np.int32)
        iso_crop = isolated[y1:y2, x1:x2]
        cv2.imwrite(f"{img_name}_{label}-{ci}.png", isolated)


image 1/1 c:\Users\Alumno\Desktop\actividad2\max.jpg: 640x384 1 person, 47.9ms
Speed: 1.8ms preprocess, 47.9ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)
